In [1]:
from pathlib import Path
import json
import re
import pandas as pd

print("Notebook working directory:")
print(Path.cwd().resolve())

Notebook working directory:
C:\Users\daoha\Downloads\IT_Tasks\BA_USDependency\eval_with_colab\dependency_analysis


In [2]:
RESULTS_DIR = Path("eval_outputs")

PREDICTION_DIR = Path("pipeline_outputs")

MODEL_RUNS = {
    "llama3.1:8b": PREDICTION_DIR / "llama3.1_8b_annotated_gt_results.json",
    "qwen3:8b": PREDICTION_DIR / "qwen3_8b_annotated_gt_results.json",
}


STAGE2_GT_PATH = Path("baseline/Ground_truth_OBD_dependency.xlsx")

STAGE2_GT_SHEET = "A_GT"

STAGE2_RESULTS_DIR = RESULTS_DIR / "stage2"
STAGE2_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

OPERATION_LABELS = [
    "CREATE",
    "READ",
    "UPDATE",
    "DELETE",
    "NON_CRUD",
]

In [3]:
for model_name, path in MODEL_RUNS.items():
    print(f"{model_name}: {'FOUND' if path.exists() else 'MISSING'} — {path}")

print(
    f"Stage 2 ground truth: "
    f"{'FOUND' if STAGE2_GT_PATH.exists() else 'MISSING'} — {STAGE2_GT_PATH}"
)

llama3.1:8b: FOUND — pipeline_outputs\llama3.1_8b_annotated_gt_results.json
qwen3:8b: FOUND — pipeline_outputs\qwen3_8b_annotated_gt_results.json
Stage 2 ground truth: FOUND — baseline\Ground_truth_OBD_dependency.xlsx


In [4]:
ID_WIDTH = 3
def normalize_id(raw) -> str:
    """Convert US_4, US-4, US-004, or 4 into US-004."""
    match = re.search(r"(\d+)", str(raw).strip())

    if not match:
        raise ValueError(f"Cannot normalize story ID: {raw!r}")

    return f"US-{int(match.group(1)):0{ID_WIDTH}d}"


def normalize_text(value: str) -> str:
    """Normalize action/entity strings for matching."""
    return " ".join(str(value).strip().casefold().split())

Load Stage 2 ground truth and validate its structure and labels.

In [5]:
def load_stage2_ground_truth(path: Path, sheet_name=0) -> pd.DataFrame:
    """Load and validate the Stage 2 ground truth DataFrame from an Excel file."""
    ground_truth = pd.read_excel(path, sheet_name=sheet_name)

    required_columns = {
        "US_ID",
        "US_Action",
        "US_Entity",
        "GT_operation_type",
    }

    missing_columns = required_columns - set(ground_truth.columns)

    if missing_columns:
        raise ValueError(
            f"Missing required Stage 2 GT columns: {sorted(missing_columns)}\n"
            f"Available columns: {list(ground_truth.columns)}"
        )

    ground_truth = ground_truth[
        ["US_ID", "US_Action", "US_Entity", "GT_operation_type"]
    ].copy()

    ground_truth["US_ID"] = ground_truth["US_ID"].apply(normalize_id)
    ground_truth["US_Action"] = ground_truth["US_Action"].astype(str).str.strip()
    ground_truth["US_Entity"] = ground_truth["US_Entity"].astype(str).str.strip()
    ground_truth["GT_operation_type"] = (
        ground_truth["GT_operation_type"]
        .astype(str)
        .str.strip()
        .str.upper()
    )

    invalid_labels = ground_truth[
        ~ground_truth["GT_operation_type"].isin(OPERATION_LABELS)
    ]

    if not invalid_labels.empty:
        raise ValueError(
            "Invalid GT operation-type labels:\n"
            + invalid_labels.to_string(index=False)
        )

    ground_truth["action_key"] = ground_truth["US_Action"].apply(normalize_text)
    ground_truth["entity_key"] = ground_truth["US_Entity"].apply(normalize_text)

    return ground_truth

In [6]:
stage2_gt = load_stage2_ground_truth(
    STAGE2_GT_PATH,
    sheet_name=STAGE2_GT_SHEET,
)

print(f"Ground-truth Stage 2 activities: {len(stage2_gt)}")
display(stage2_gt)

Ground-truth Stage 2 activities: 34


,US_ID,US_Action,US_Entity,GT_operation_type,action_key,entity_key
0,US-001,sign up,email,CREATE,sign up,email
1,US-001,sign up,existing Google account,CREATE,sign up,existing google account
2,US-001,sign up,existing Apple account,CREATE,sign up,existing apple account
3,US-002,log in,account,READ,log in,account
4,US-002,log out,account,NON_CRUD,log out,account
5,US-003,walk through,"short, guided tutorial",READ,walk through,"short, guided tutorial"
6,US-004,import,location,CREATE,import,location
7,US-005,create,named lists,CREATE,create,named lists
8,US-005,add,saved spots,CREATE,add,saved spots
9,US-006,attach,private notes,CREATE,attach,private notes


Load Stage 2 results from JSON files and validate their structure and labels.

In [7]:
def load_json_results(path: Path) -> list[dict]:
    """Load and validate the Stage 2 results from a JSON file."""
    with open(path, "r", encoding="utf-8") as file:
        return json.load(file)

In [8]:
def extract_stage2_predictions(
    pipeline_results: list[dict],
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Extract one unique Stage 2 prediction per: (US_ID, action, entity)
    Returns:
    - predictions: deduplicated classified activities
    - conflicts: activities that received conflicting operation labels
    """
    predictions: dict[tuple[str, str, str], dict] = {}
    conflicts: list[dict] = []

    for pair_result in pipeline_results:
        step2 = pair_result.get("step_2_classify", {})

        story_a_id = step2.get("story_a_id")
        story_b_id = step2.get("story_b_id")

        for match in step2.get("classified_matches", []):
            activities = [
                (story_a_id, match["activity_a"]),
                (story_b_id, match["activity_b"]),
            ]

            for story_id, activity in activities:
                story_id = normalize_id(story_id)

                action = activity["action"]
                entity = activity["entity"]
                operation_type = activity["operation_type"]

                key = (
                    story_id,
                    normalize_text(action),
                    normalize_text(entity),
                )

                record = {
                    "US_ID": story_id,
                    "US_Action": action,
                    "US_Entity": entity,
                    "predicted_operation_type": operation_type,
                    "action_key": normalize_text(action),
                    "entity_key": normalize_text(entity),
                }

                if key not in predictions:
                    predictions[key] = record

                elif predictions[key]["predicted_operation_type"] != operation_type:
                    conflicts.append(
                        {
                            "US_ID": story_id,
                            "US_Action": action,
                            "US_Entity": entity,
                            "first_operation_type": predictions[key][
                                "predicted_operation_type"
                            ],
                            "conflicting_operation_type": operation_type,
                        }
                    )

    prediction_df = pd.DataFrame(predictions.values())

    conflict_df = pd.DataFrame(
        conflicts,
        columns=[
            "US_ID",
            "US_Action",
            "US_Entity",
            "first_operation_type",
            "conflicting_operation_type",
        ],
    )

    return prediction_df, conflict_df

In [9]:
predictions_by_model = {}
conflicts_by_model = {}

for model_name, result_path in MODEL_RUNS.items():
    pipeline_results = load_json_results(result_path)

    predictions, conflicts = extract_stage2_predictions(pipeline_results)

    predictions_by_model[model_name] = predictions
    conflicts_by_model[model_name] = conflicts

    print(f"\n{model_name}")
    print(f"Unique Stage 2 predictions: {len(predictions)}")
    print(f"Conflicting operation labels: {len(conflicts)}")

    if not conflicts.empty:
        display(conflicts)


llama3.1:8b
Unique Stage 2 predictions: 18
Conflicting operation labels: 0

qwen3:8b
Unique Stage 2 predictions: 16
Conflicting operation labels: 0


In [10]:
print(predictions_by_model)

{'llama3.1:8b':      US_ID     US_Action              US_Entity predicted_operation_type  \
0   US-004        import               location                   CREATE   
1   US-005           add            saved spots                   CREATE   
2   US-009     selecting               location                     READ   
3   US-011           see            saved spots                     READ   
4   US-009  manually add                   spot                   CREATE   
5   US-005        create            named lists                   CREATE   
6   US-010        change                   name                   UPDATE   
7   US-014        select                   list                     READ   
8   US-016          drag                  stops                 NON_CRUD   
9   US-017        adjust                  spots                   UPDATE   
10  US-009      entering                   name                   UPDATE   
11  US-014        select            destination                     READ

In [37]:
def evaluate_stage2(
    predictions: pd.DataFrame,
    ground_truth: pd.DataFrame,
    model_name: str,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Conditional Step 2 evaluation.
    Activities missing from pipeline Step 2 are not counted as operation-label false negatives. They are measured through coverage instead.
    """
    evaluation = ground_truth.merge(
        predictions[
            [
                "US_ID",
                "action_key",
                "entity_key",
                "predicted_operation_type",
            ]
        ],
        how="left",
        on=["US_ID", "action_key", "entity_key"],
    )

    evaluation["covered"] = evaluation["predicted_operation_type"].notna()

    covered = evaluation[evaluation["covered"]].copy()

    covered["correct"] = (
        covered["GT_operation_type"]
        == covered["predicted_operation_type"]
    )

    total_gt = len(evaluation)
    total_covered = len(covered)
    correct = int(covered["correct"].sum())
    incorrect = total_covered - correct

    coverage = total_covered / total_gt if total_gt else 0.0
    accuracy = correct / total_covered if total_covered else 0.0

    # For single-label multiclass classification:
    # every incorrect label is one micro FP and one micro FN.
    overall = pd.DataFrame(
        [
            {
                "Model": model_name,
                "GT activities": total_gt,
                "Covered activities": total_covered,
                "Coverage": round(coverage, 4),
                "Correct": correct,
                "Incorrect": incorrect,
                "Micro TP": correct,
                "Micro FP": incorrect,
                "Micro FN": incorrect,
                "Accuracy / Micro F1": round(accuracy, 4),
            }
        ]
    )

    per_class_rows = []

    for label in OPERATION_LABELS:
        tp = int(
            (
                (covered["GT_operation_type"] == label)
                & (covered["predicted_operation_type"] == label)
            ).sum()
        )

        fp = int(
            (
                (covered["GT_operation_type"] != label)
                & (covered["predicted_operation_type"] == label)
            ).sum()
        )

        fn = int(
            (
                (covered["GT_operation_type"] == label)
                & (covered["predicted_operation_type"] != label)
            ).sum()
        )

        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = (
            2 * precision * recall / (precision + recall)
            if (precision + recall)
            else 0.0
        )

        per_class_rows.append(
            {
                "Model": model_name,
                "Operation type": label,
                "TP": tp,
                "FP": fp,
                "FN": fn,
                "Precision": round(precision, 4),
                "Recall": round(recall, 4),
                "F1": round(f1, 4),
                "Support": int(
                    (covered["GT_operation_type"] == label).sum()
                ),
            }
        )

    per_class = pd.DataFrame(per_class_rows)

    return overall, per_class, evaluation

In [38]:
overall_results = []
per_class_results = []
details_by_model = {}

for model_name, predictions in predictions_by_model.items():
    overall, per_class, details = evaluate_stage2(
        predictions=predictions,
        ground_truth=stage2_gt,
        model_name=model_name,
    )

    overall_results.append(overall)
    per_class_results.append(per_class)
    details_by_model[model_name] = details

In [41]:
print(overall_results)

[         Model  GT activities  Covered activities  Coverage  Correct  \
0  llama3.1:8b             34                  18    0.5294       14   

   Incorrect  Micro TP  Micro FP  Micro FN  Accuracy / Micro F1  
0          4        14         4         4               0.7778  ,       Model  GT activities  Covered activities  Coverage  Correct  Incorrect  \
0  qwen3:8b             34                  16    0.4706       15          1   

   Micro TP  Micro FP  Micro FN  Accuracy / Micro F1  
0        15         1         1               0.9375  ]


In [39]:
def show_stage2_confusion_matrix(
    evaluation_details: pd.DataFrame,
    model_name: str,
) -> pd.DataFrame:
    """
    Display a Stage 2 confusion matrix for activities that reached Step 2.

    Rows = ground-truth operation type
    Columns = predicted operation type
    """
    covered = evaluation_details[ evaluation_details["covered"]].copy()

    matrix = pd.crosstab(covered["GT_operation_type"],covered["predicted_operation_type"],)

    matrix = matrix.reindex(index=OPERATION_LABELS,columns=OPERATION_LABELS,fill_value=0,)

    matrix.index.name = "Ground truth"
    matrix.columns.name = "Predicted"

    print(f"===== Stage 2 Confusion Matrix: {model_name} =====")
    print(f"Covered activities included: {len(covered)}")
    display(matrix)

    return matrix

In [40]:
llama_confusion_matrix = show_stage2_confusion_matrix(
    evaluation_details=details_by_model["llama3.1:8b"],
    model_name="llama3.1:8b",
)

qwen_confusion_matrix = show_stage2_confusion_matrix(
    evaluation_details=details_by_model["qwen3:8b"],
    model_name="qwen3:8b",
)

===== Stage 2 Confusion Matrix: llama3.1:8b =====
Covered activities included: 18


Predicted,CREATE,READ,UPDATE,DELETE,NON_CRUD
Ground truth,,,,,
CREATE,4,0,2,0,0
READ,0,7,0,0,0
UPDATE,0,0,3,1,1
DELETE,0,0,0,0,0
NON_CRUD,0,0,0,0,0


===== Stage 2 Confusion Matrix: qwen3:8b =====
Covered activities included: 16


Predicted,CREATE,READ,UPDATE,DELETE,NON_CRUD
Ground truth,,,,,
CREATE,5,1,0,0,0
READ,0,7,0,0,0
UPDATE,0,0,3,0,0
DELETE,0,0,0,0,0
NON_CRUD,0,0,0,0,0


In [23]:
def save_confusion_matrix_latex(
    matrix: pd.DataFrame,
    model_name: str,
    output_dir: Path,
) -> str:
    """
    Convert one Step 2 confusion matrix to LaTex and save it to a .tex file.
    """
    safe_model_name = (
        model_name
        .replace(":", "_")
        .replace(".", "_")
    )

    latex = matrix.to_latex(
        escape=False,
        caption=(
            f"Stage 2 CRUD operation-type confusion matrix for "
            f"{model_name}. Rows are ground-truth labels; columns are "
            f"predicted labels."
        ),
        label=f"tab:stage2_confusion_{safe_model_name}",
        position="htbp",
    )

    output_path = output_dir / f"{safe_model_name}_stage2_confusion_matrix.tex"

    output_path.write_text(
        latex,
        encoding="utf-8",
    )

    print(f"Saved: {output_path}")

    return latex

In [24]:
llama_confusion_latex = save_confusion_matrix_latex(
    matrix=llama_confusion_matrix,
    model_name="llama3.1:8b",
    output_dir=STAGE2_RESULTS_DIR,
)

qwen_confusion_latex = save_confusion_matrix_latex(
    matrix=qwen_confusion_matrix,
    model_name="qwen3:8b",
    output_dir=STAGE2_RESULTS_DIR,
)

Saved: eval_outputs\stage2\llama3_1_8b_stage2_confusion_matrix.tex
Saved: eval_outputs\stage2\qwen3_8b_stage2_confusion_matrix.tex


In [25]:
print(llama_confusion_latex)
print(qwen_confusion_latex)

\begin{table}[htbp]
\caption{Stage 2 CRUD operation-type confusion matrix for llama3.1:8b. Rows are ground-truth labels; columns are predicted labels.}
\label{tab:stage2_confusion_llama3_1_8b}
\begin{tabular}{lrrrrr}
\toprule
Predicted & CREATE & READ & UPDATE & DELETE & NON_CRUD \\
Ground truth &  &  &  &  &  \\
\midrule
CREATE & 4 & 1 & 2 & 0 & 0 \\
READ & 0 & 6 & 0 & 0 & 0 \\
UPDATE & 0 & 0 & 3 & 1 & 1 \\
DELETE & 0 & 0 & 0 & 0 & 0 \\
NON_CRUD & 0 & 0 & 0 & 0 & 0 \\
\bottomrule
\end{tabular}
\end{table}

\begin{table}[htbp]
\caption{Stage 2 CRUD operation-type confusion matrix for qwen3:8b. Rows are ground-truth labels; columns are predicted labels.}
\label{tab:stage2_confusion_qwen3_8b}
\begin{tabular}{lrrrrr}
\toprule
Predicted & CREATE & READ & UPDATE & DELETE & NON_CRUD \\
Ground truth &  &  &  &  &  \\
\midrule
CREATE & 5 & 2 & 0 & 0 & 0 \\
READ & 0 & 6 & 0 & 0 & 0 \\
UPDATE & 0 & 0 & 3 & 0 & 0 \\
DELETE & 0 & 0 & 0 & 0 & 0 \\
NON_CRUD & 0 & 0 & 0 & 0 & 0 \\
\bottomrule
\end{ta